# ?? Inferensi Risiko Diabetes + Sistem Rekomendasi
## Model XGBoost ? Prediksi Risiko Prediabetes pada Usia Muda
---
### ?? Ringkasan Notebook

| Item | Keterangan |
|---|---|
| **Model** | XGBoost (`model_xgboost_risk_level.pkl`) |
| **Jumlah Fitur** | 12 fitur |
| **Kategori Scoring** | Dihitung manual dari `Risk_Score` revisi |
| **Output Model** | Tidak Berisiko / Sedang / Tinggi |
| **Keyakinan Model** | Probabilitas tertinggi dari `predict_proba()`; bukan probabilitas medis |
| **Rekomendasi** | Berdasarkan faktor risiko input user |

### ?? Perubahan Input dari Versi Sebelumnya

| Fitur | Sebelumnya | Sekarang |
|---|---|---|
| `BMI` | Input angka langsung | Dihitung dari BB (kg) + TB (cm) |
| `Genetic_Risk_Score` | Angka 1?10 | 3 pilihan deskriptif |
| `Sleep_Hours` | Angka jam | 4 pilihan durasi |
| `Stress_Level` | Angka 1?10 | 3 pilihan deskriptif |

## STEP 1 — Import Library

In [2]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.scoring import RISK_LABELS, calculate_risk_score, create_risk_level

print('? Library berhasil diimport!')

? Library berhasil diimport!


## STEP 2 — Load Model

> ⚠️ Pastikan file `model_xgboost_risk_level.pkl` sudah diupload ke Colab sebelum menjalankan cell ini.

In [3]:
model_path = PROJECT_ROOT / 'models' / 'model_xgboost_risk_level.pkl'
model_artifact = joblib.load(model_path)

feature_names = [
    'Age', 'BMI', 'HbA1c', 'Fasting_Blood_Sugar',
    'Genetic_Risk_Score', 'Family_History_Diabetes',
    'Physical_Activity_Level', 'Dietary_Habits',
    'Smoking', 'Alcohol_Consumption', 'Sleep_Hours', 'Stress_Level'
]

if isinstance(model_artifact, dict):
    model_risk = model_artifact['model']
    feature_names = model_artifact.get('feature_columns', feature_names)
    label_map = model_artifact.get('target_classes', RISK_LABELS)
    importance_dict = model_artifact.get(
        'feature_importance',
        dict(zip(feature_names, model_risk.feature_importances_))
    )
else:
    model_risk = model_artifact
    label_map = RISK_LABELS
    importance_dict = dict(zip(feature_names, model_risk.feature_importances_))

print('? Model berhasil di-load!')
print()
print('Feature Importance (diurutkan):')
for feat, imp in sorted(importance_dict.items(), key=lambda x: x[1], reverse=True):
    bar = '?' * int(float(imp) * 100)
    print(f'  {feat:<25}: {float(imp):.4f}  {bar}')

? Model berhasil di-load!

Feature Importance (diurutkan):
  HbA1c                    : 0.1381  ?????????????
  BMI                      : 0.1357  ?????????????
  Genetic_Risk_Score       : 0.1138  ???????????
  Fasting_Blood_Sugar      : 0.1116  ???????????
  Family_History_Diabetes  : 0.0877  ????????
  Physical_Activity_Level  : 0.0742  ???????
  Smoking                  : 0.0703  ???????
  Dietary_Habits           : 0.0697  ??????
  Age                      : 0.0619  ??????
  Stress_Level             : 0.0590  ?????
  Alcohol_Consumption      : 0.0488  ????
  Sleep_Hours              : 0.0292  ??


## STEP 3 ? Fungsi Kalkulasi & Mapping Input

Fungsi berikut mengubah input user yang lebih mudah dipahami menjadi nilai numerik yang dibutuhkan model.

| Fungsi | Kegunaan |
|---|---|
| `hitung_bmi()` | Menghitung BMI dari berat & tinggi badan |
| `map_riwayat_diabetes_keluarga()` | Mengubah satu pilihan riwayat keluarga menjadi `family_history` dan `genetic_risk_score` |
| `mapping_genetic()` | Kompatibilitas untuk pemanggilan lama berbasis `pilihan_genetic` |
| `mapping_sleep()` | Mengubah pilihan durasi tidur ke angka jam |
| `mapping_stress()` | Mengubah pilihan tingkat stres ke angka 1?10 |

In [4]:
def hitung_bmi(berat_kg, tinggi_cm):
    """
    Menghitung BMI dari berat badan (kg) dan tinggi badan (cm).
    Rumus: BMI = BB / (TB dalam meter)^2
    """
    tinggi_m = tinggi_cm / 100
    bmi = berat_kg / (tinggi_m ** 2)

    if bmi < 18.5:
        kategori = 'Underweight'
    elif bmi < 25:
        kategori = 'Normal'
    elif bmi < 30:
        kategori = 'Overweight'
    else:
        kategori = 'Obesitas'

    return round(bmi, 2), kategori


FAMILY_GENETIC_MAPPING = {
    "Tidak ada keluarga diabetes": {
        "family_history": 0,
        "genetic_risk_score": 1,
        "pilihan_genetic": "Tidak ada keluarga diabetes",
    },
    "Tidak tahu / gunakan nilai default": {
        "family_history": 0,
        "genetic_risk_score": 5,
        "pilihan_genetic": "Tidak tahu / gunakan nilai default",
    },
    "Ada pada keluarga besar": {
        "family_history": 1,
        "genetic_risk_score": 6,
        "pilihan_genetic": "Ada pada keluarga besar",
    },
    "Ada pada ayah/ibu/saudara kandung": {
        "family_history": 1,
        "genetic_risk_score": 8,
        "pilihan_genetic": "Ada (ayah/ibu/saudara kandung)",
    },
}


def map_riwayat_diabetes_keluarga(riwayat_diabetes_keluarga: str) -> dict:
    return FAMILY_GENETIC_MAPPING.get(
        riwayat_diabetes_keluarga,
        FAMILY_GENETIC_MAPPING["Tidak tahu / gunakan nilai default"]
    )


def mapping_genetic(pilihan):
    """
    Kompatibilitas untuk pemanggilan lama berbasis pilihan_genetic.
    Mapping utama yang disarankan adalah FAMILY_GENETIC_MAPPING.
    """
    mapping = {
        'Tidak ada': 1,
        'Tidak ada keluarga diabetes': 1,
        'Tidak tahu / gunakan nilai default': 5,
        'Ada (paman/bibi/kakek/nenek)': 6,
        'Ada pada keluarga besar': 6,
        'Ada pada ayah/ibu/saudara kandung': 8,
        'Ada (ayah/ibu/saudara kandung)': 8,
    }
    return mapping.get(pilihan, FAMILY_GENETIC_MAPPING["Tidak tahu / gunakan nilai default"]["genetic_risk_score"])


def mapping_sleep(pilihan):
    """
    Mengubah pilihan durasi tidur menjadi angka jam.
    Pilihan:
      - 'Kurang dari 5 jam'  ? 4.5
      - '5-6 jam'            ? 5.5
      - '7-8 jam (ideal)'    ? 7.5
      - 'Lebih dari 8 jam'   ? 9.0
    """
    mapping = {
        'Kurang dari 5 jam':   4.5,
        '5-6 jam':             5.5,
        '7-8 jam (ideal)':     7.5,
        'Lebih dari 8 jam':    9.0
    }
    return mapping[pilihan]


def mapping_stress(pilihan):
    """
    Mengubah pilihan tingkat stres menjadi angka (1?10).
    Mendukung label lama notebook dan label baru aplikasi Streamlit.
    """
    mapping = {
        'Jarang stres': 2,
        'Hampir tidak pernah stres': 2,
        'Kadang-kadang stres': 5,
        'Sering stres': 8,
        'Sering stres / sulit dikendalikan': 8,
        'Sangat sering stres': 10,
    }
    return mapping[pilihan]


print('? Fungsi kalkulasi & mapping berhasil didefinisikan!')

? Fungsi kalkulasi & mapping berhasil didefinisikan!


## STEP 4 - Fungsi Sistem Rekomendasi

Rekomendasi dibangkitkan berdasarkan kondisi tiap fitur input user. Kategori risiko utama dihitung dengan sistem scoring manual revisi.

### Logika Keyakinan Model

`Keyakinan Model = probabilitas tertinggi dari predict_proba() x 100`

Nilai ini menunjukkan tingkat keyakinan model terhadap hasil prediksi, bukan probabilitas medis bahwa pengguna mengalami diabetes.

In [5]:
def generate_recommendations(data, risk_label, importance_dict):
    """
    Membangkitkan rekomendasi personal berdasarkan kondisi fitur input.
    Diurutkan dari fitur dengan importance tertinggi ke terendah.

    Args:
        data          : dict berisi nilai fitur user (sudah dalam bentuk numerik)
        risk_label    : label prediksi ('Tidak Berisiko', 'Sedang', 'Tinggi')
        importance_dict: dict {nama_fitur: importance_score} dari model

    Returns:
        list of str: rekomendasi yang sudah diurutkan
    """
    recs = []  # list of (teks_rekomendasi, importance_score)

    # ── HbA1c ────────────────────────────────────────────
    if data['HbA1c'] >= 6.5:
        recs.append((
            '🩸 HbA1c kamu ≥6.5%, ini indikasi diabetes. Segera konsultasikan ke dokter untuk evaluasi lebih lanjut.',
            importance_dict['HbA1c']
        ))
    elif data['HbA1c'] >= 5.7:
        recs.append((
            '🩸 HbA1c kamu di rentang prediabetes (5.7–6.4%). Kurangi konsumsi gula dan karbohidrat sederhana, pantau secara berkala.',
            importance_dict['HbA1c']
        ))

    # ── BMI ──────────────────────────────────────────────
    if data['BMI'] >= 30:
        recs.append((
            '⚖️ BMI kamu masuk kategori Obesitas. Targetkan penurunan berat badan bertahap dengan diet seimbang dan olahraga rutin. Konsultasikan ke ahli gizi.',
            importance_dict['BMI']
        ))
    elif data['BMI'] >= 25:
        recs.append((
            '⚖️ BMI kamu masuk kategori Overweight. Jaga pola makan dan mulai olahraga rutin untuk mencapai BMI ideal (18.5–24.9).',
            importance_dict['BMI']
        ))

    # ── Fasting Blood Sugar ───────────────────────────────
    if data['Fasting_Blood_Sugar'] >= 126:
        recs.append((
            '🍬 Gula darah puasa kamu ≥126 mg/dL, indikasi diabetes. Segera periksakan ke dokter.',
            importance_dict['Fasting_Blood_Sugar']
        ))
    elif data['Fasting_Blood_Sugar'] >= 100:
        recs.append((
            '🍬 Gula darah puasa kamu 100–125 mg/dL (prediabetes). Kurangi karbohidrat sederhana dan gula tambahan.',
            importance_dict['Fasting_Blood_Sugar']
        ))

    # ── Family History ────────────────────────────────────
    if data['Family_History_Diabetes'] == 1:
        recs.append((
            '👨\u200d👩\u200d👧 Kamu memiliki riwayat keluarga diabetes. Lakukan skrining gula darah rutin minimal 1x/tahun.',
            importance_dict['Family_History_Diabetes']
        ))

    # ── Physical Activity ─────────────────────────────────
    if data['Physical_Activity_Level'] == 0:
        recs.append((
            '🏃 Kamu tergolong sedentary. Mulai lakukan minimal 150 menit aktivitas aerobik sedang per minggu (jalan cepat, bersepeda, berenang).',
            importance_dict['Physical_Activity_Level']
        ))
    elif data['Physical_Activity_Level'] == 1:
        recs.append((
            '🚶 Aktivitas fisik kamu cukup tapi belum optimal. Tingkatkan ke level aktif dengan target 150–300 menit/minggu.',
            importance_dict['Physical_Activity_Level']
        ))

    # ── Genetic Risk Score ────────────────────────────────
    if data['Genetic_Risk_Score'] >= 8:
        recs.append((
            '🧬 Risiko genetik kamu tinggi. Perhatikan lebih ketat pola makan dan gaya hidup sebagai bentuk pencegahan dini.',
            importance_dict['Genetic_Risk_Score']
        ))
    elif data['Genetic_Risk_Score'] >= 5:
        recs.append((
            '🧬 Risiko genetik kamu sedang. Tetap jaga gaya hidup sehat untuk meminimalkan risiko.',
            importance_dict['Genetic_Risk_Score']
        ))

    # ── Dietary Habits ────────────────────────────────────
    if data['Dietary_Habits'] == 0:
        recs.append((
            '🥗 Pola makan kamu tidak sehat. Terapkan pola makan seimbang: perbanyak sayur, buah, biji-bijian, dan kurangi lemak jenuh.',
            importance_dict['Dietary_Habits']
        ))
    elif data['Dietary_Habits'] == 1:
        recs.append((
            '🥦 Pola makan kamu cukup, tapi bisa lebih baik. Konsistenkan asupan bergizi setiap hari.',
            importance_dict['Dietary_Habits']
        ))

    # ── Stress Level ──────────────────────────────────────
    if data['Stress_Level'] >= 7:
        recs.append((
            '🧘 Tingkat stres kamu tinggi. Stres kronis menghambat sensitivitas insulin. Coba meditasi, yoga, atau konseling.',
            importance_dict['Stress_Level']
        ))
    elif data['Stress_Level'] >= 4:
        recs.append((
            '😌 Stres kamu di level sedang. Kelola dengan aktivitas relaksasi seperti jalan santai atau hobi.',
            importance_dict['Stress_Level']
        ))

    # ── Sleep Hours ───────────────────────────────────────
    if data['Sleep_Hours'] < 6:
        recs.append((
            '😴 Durasi tidur kamu kurang dari 6 jam. Kurang tidur meningkatkan risiko diabetes 28%. Targetkan 7–9 jam/malam.',
            importance_dict['Sleep_Hours']
        ))

    # ── Smoking ───────────────────────────────────────────
    if data['Smoking'] == 1:
        recs.append((
            '🚭 Kamu merokok. Merokok meningkatkan risiko diabetes tipe 2 sebesar 30–40%. Sangat disarankan untuk berhenti.',
            importance_dict['Smoking']
        ))

    # ── Alcohol ───────────────────────────────────────────
    if data['Alcohol_Consumption'] == 1:
        recs.append((
            '🍺 Kamu mengonsumsi alkohol. Alkohol berlebihan mengganggu metabolisme glukosa. Batasi atau hentikan.',
            importance_dict['Alcohol_Consumption']
        ))

    # Sort by importance (tertinggi duluan)
    recs.sort(key=lambda x: x[1], reverse=True)

    # Rekomendasi umum berdasarkan risk label (selalu di akhir)
    if risk_label == 'Tinggi':
        recs.append(('🏥 PRIORITAS: Segera konsultasikan kondisi kamu ke dokter atau ahli gizi untuk evaluasi menyeluruh.', 0))
    elif risk_label == 'Sedang':
        recs.append(('📋 Lakukan pemeriksaan kesehatan berkala setiap 6 bulan untuk memantau perkembangan kondisi kamu.', 0))
    else:
        recs.append(('✅ Pertahankan gaya hidup sehat kamu! Tetap lakukan pemeriksaan tahunan sebagai deteksi dini.', 0))

    return [r[0] for r in recs]


print('✅ Fungsi rekomendasi berhasil didefinisikan!')

✅ Fungsi rekomendasi berhasil didefinisikan!


## STEP 5 — Fungsi Utama Prediksi

Fungsi `prediksi_diabetes()` menerima input user, menjalankan kalkulasi & mapping, melakukan prediksi, dan menampilkan hasil lengkap beserta rekomendasi.

In [6]:
def prediksi_diabetes(
    usia,
    berat_kg,
    tinggi_cm,
    hba1c,
    fasting_blood_sugar,
    riwayat_diabetes_keluarga=None,
    pilihan_genetic=None,
    family_history=None,
    physical_activity=1,
    dietary_habits=1,
    smoking=0,
    alcohol=0,
    pilihan_sleep="7-8 jam (ideal)",
    pilihan_stress="Kadang-kadang stres",
):
    """
    Fungsi utama prediksi risiko diabetes.

    Parameter utama untuk riwayat keluarga adalah `riwayat_diabetes_keluarga`.
    Parameter lama `pilihan_genetic` dan `family_history` tetap didukung untuk compatibility.

    Returns:
        dict berisi skor manual, kategori scoring, kategori model, keyakinan model,
        BMI, probabilitas kelas, rekomendasi, dan detail teknis mapping keluarga.
    """
    bmi, kategori_bmi = hitung_bmi(berat_kg, tinggi_cm)

    if riwayat_diabetes_keluarga is not None:
        family_config = map_riwayat_diabetes_keluarga(riwayat_diabetes_keluarga)
        family_history = family_config["family_history"]
        pilihan_genetic = family_config["pilihan_genetic"]
        genetic_risk_score = family_config["genetic_risk_score"]
    else:
        if pilihan_genetic is None:
            pilihan_genetic = "Tidak tahu / gunakan nilai default"
        if family_history is None:
            family_history = 0
        genetic_risk_score = mapping_genetic(pilihan_genetic)
        riwayat_diabetes_keluarga = pilihan_genetic

    sleep_hours = mapping_sleep(pilihan_sleep)
    stress_level = mapping_stress(pilihan_stress)

    data = {
        'Age': usia,
        'BMI': bmi,
        'HbA1c': hba1c,
        'Fasting_Blood_Sugar': fasting_blood_sugar,
        'Genetic_Risk_Score': genetic_risk_score,
        'Family_History_Diabetes': family_history,
        'Physical_Activity_Level': physical_activity,
        'Dietary_Habits': dietary_habits,
        'Smoking': smoking,
        'Alcohol_Consumption': alcohol,
        'Sleep_Hours': sleep_hours,
        'Stress_Level': stress_level,
    }

    risk_score = calculate_risk_score(data)
    scoring_label = create_risk_level(risk_score)

    df_input = pd.DataFrame([data])[feature_names]
    pred = model_risk.predict(df_input)[0]
    pred_proba = model_risk.predict_proba(df_input)[0] if hasattr(model_risk, 'predict_proba') else None
    model_label = label_map[int(pred)] if isinstance(label_map, dict) and int(pred) in label_map else str(pred)
    model_confidence = float(pred_proba.max()) * 100 if pred_proba is not None else None

    if pred_proba is not None:
        probabilities = {
            label_map[int(cls)] if isinstance(label_map, dict) and int(cls) in label_map else str(cls): round(float(prob), 4)
            for cls, prob in zip(model_risk.classes_, pred_proba)
        }
    else:
        probabilities = {}

    recommendations = generate_recommendations(data, scoring_label, importance_dict)

    print('=' * 60)
    print('          HASIL PREDIKSI RISIKO DIABETES')
    print('=' * 60)
    print(f'  Usia                       : {usia} tahun')
    print(f'  Berat Badan                : {berat_kg} kg')
    print(f'  Tinggi Badan               : {tinggi_cm} cm')
    print(f'  BMI                        : {bmi} ({kategori_bmi})')
    print(f'  HbA1c                      : {hba1c}%')
    print(f'  Gula Darah Puasa           : {fasting_blood_sugar} mg/dL')
    print(f'  Riwayat Diabetes Keluarga  : {riwayat_diabetes_keluarga}')
    print()
    print(f'  Skor Risiko                : {risk_score}')
    print(f'  Kategori Scoring           : {scoring_label}')
    print(f'  Kategori Prediksi Model    : {model_label}')
    if model_confidence is not None:
        print(f'  Keyakinan Model            : {model_confidence:.2f}%')
        print('  Catatan                    : Keyakinan model bukan probabilitas medis.')
    if scoring_label != model_label:
        print('  PERINGATAN                 : Hasil model berbeda dari kategori scoring.')
    print()
    if probabilities:
        print('  Probabilitas per kelas:')
        for name, prob in probabilities.items():
            bar = '?' * int(prob * 30)
            print(f'    {name:<18}: {prob:.2%}  {bar}')
        print()
    print('=' * 60)
    print('          REKOMENDASI PERSONAL')
    print('=' * 60)
    for i, rec in enumerate(recommendations, 1):
        print(f'  {i}. {rec}')
    print('=' * 60)

    return {
        'risk_score': round(risk_score),
        'scoring_label': scoring_label,
        'model_risk_label': model_label,
        'model_confidence_pct': round(model_confidence, 2) if model_confidence is not None else None,
        'probabilities': probabilities,
        'proba': probabilities,
        'bmi': bmi,
        'bmi_category': kategori_bmi,
        'kategori_bmi': kategori_bmi,
        'recommendations': recommendations,
        'riwayat_diabetes_keluarga': riwayat_diabetes_keluarga,
        'family_history': family_history,
        'genetic_risk_score': genetic_risk_score,
        'pilihan_genetic': pilihan_genetic,
        'faktor_risiko': data,
    }


print('? Fungsi prediksi berhasil didefinisikan!')

? Fungsi prediksi berhasil didefinisikan!


## STEP 6 ? Panduan Nilai Input

Sebelum menjalankan prediksi, pastikan nilai input sesuai panduan berikut:

| Parameter | Tipe | Nilai yang Valid |
|---|---|---|
| `usia` | int | 15 ? 25 tahun |
| `berat_kg` | float | Berat badan dalam kg |
| `tinggi_cm` | float | Tinggi badan dalam cm |
| `hba1c` | float | 4.0 ? 10.0 (%) |
| `fasting_blood_sugar` | float | 70 ? 180 (mg/dL) |
| `riwayat_diabetes_keluarga` | str | `'Tidak ada keluarga diabetes'` / `'Tidak tahu / gunakan nilai default'` / `'Ada pada keluarga besar'` / `'Ada pada ayah/ibu/saudara kandung'` |
| `physical_activity` | int | `0` = Sedentary, `1` = Moderate, `2` = Active |
| `dietary_habits` | int | `0` = Unhealthy, `1` = Moderate, `2` = Healthy |
| `smoking` | int | `1` = Ya, `0` = Tidak |
| `alcohol` | int | `1` = Ya, `0` = Tidak |
| `pilihan_sleep` | str | `'Kurang dari 5 jam'` / `'5-6 jam'` / `'7-8 jam (ideal)'` / `'Lebih dari 8 jam'` |
| `pilihan_stress` | str | `'Jarang stres'` / `'Kadang-kadang stres'` / `'Sering stres'` / `'Sangat sering stres'` |

`riwayat_diabetes_keluarga` adalah input ramah pengguna. Sistem akan otomatis mengubah pilihan ini menjadi `family_history` dan `genetic_risk_score` untuk kebutuhan scoring dan model.

Pilihan `riwayat_diabetes_keluarga`:

1. `Tidak ada keluarga diabetes`
2. `Tidak tahu / gunakan nilai default`
3. `Ada pada keluarga besar`
4. `Ada pada ayah/ibu/saudara kandung`

Parameter lama `pilihan_genetic` dan `family_history` masih didukung untuk kompatibilitas, tetapi bukan lagi input utama yang disarankan.

## STEP 7 — Jalankan Prediksi

Ganti nilai-nilai di bawah sesuai data yang ingin diprediksi.

In [7]:
# ??????????????????????????????????????????????????????????
# Ganti nilai di sini sesuai data user yang ingin diprediksi
# ??????????????????????????????????????????????????????????

hasil = prediksi_diabetes(
    usia                       = 23,
    berat_kg                   = 75,
    tinggi_cm                  = 170,
    hba1c                      = 5.9,
    fasting_blood_sugar        = 105,
    riwayat_diabetes_keluarga  = 'Ada pada ayah/ibu/saudara kandung',
    physical_activity          = 1,      # 0=Sedentary, 1=Moderate, 2=Active
    dietary_habits             = 1,      # 0=Unhealthy, 1=Moderate, 2=Healthy
    smoking                    = 0,
    alcohol                    = 0,
    pilihan_sleep              = '7-8 jam (ideal)',
    pilihan_stress             = 'Kadang-kadang stres'
)

          HASIL PREDIKSI RISIKO DIABETES
  Usia                       : 23 tahun
  Berat Badan                : 75 kg
  Tinggi Badan               : 170 cm
  BMI                        : 25.95 (Overweight)
  HbA1c                      : 5.9%
  Gula Darah Puasa           : 105 mg/dL
  Riwayat Diabetes Keluarga  : Ada pada ayah/ibu/saudara kandung

  Skor Risiko                : 14
  Kategori Scoring           : Tinggi
  Kategori Prediksi Model    : Tinggi
  Keyakinan Model            : 78.58%
  Catatan                    : Keyakinan model bukan probabilitas medis.

  Probabilitas per kelas:
    Tidak Berisiko    : 0.16%  
    Sedang            : 21.26%  ??????
    Tinggi            : 78.58%  ???????????????????????

          REKOMENDASI PERSONAL
  1. 🩸 HbA1c kamu di rentang prediabetes (5.7–6.4%). Kurangi konsumsi gula dan karbohidrat sederhana, pantau secara berkala.
  2. ⚖️ BMI kamu masuk kategori Overweight. Jaga pola makan dan mulai olahraga rutin untuk mencapai BMI ideal (18.5–24.

## STEP 8 — Test dengan Beberapa Kasus Sekaligus

Cell ini menjalankan 4 skenario berbeda untuk memverifikasi bahwa sistem rekomendasi dan persentase risiko bekerja dengan benar.

In [8]:
test_cases = [
    {
        "nama": "TEST 1 - Contoh skor 7 harus Tidak Berisiko",
        "expected_score": 7,
        "expected_label": "Tidak Berisiko",
        "params": dict(
            usia=23,
            berat_kg=60,
            tinggi_cm=165,
            hba1c=5.7,
            fasting_blood_sugar=100,
            riwayat_diabetes_keluarga="Tidak ada keluarga diabetes",
            physical_activity=1,
            dietary_habits=1,
            smoking=0,
            alcohol=0,
            pilihan_sleep="7-8 jam (ideal)",
            pilihan_stress="Kadang-kadang stres",
        ),
    },
    {
        "nama": "TEST 2 - Faktor sedang harus minimal Sedang",
        "expected_min_label": "Sedang",
        "params": dict(
            usia=23,
            berat_kg=78,
            tinggi_cm=170,
            hba1c=6.0,
            fasting_blood_sugar=110,
            riwayat_diabetes_keluarga="Ada pada ayah/ibu/saudara kandung",
            physical_activity=0,
            dietary_habits=1,
            smoking=0,
            alcohol=0,
            pilihan_sleep="7-8 jam (ideal)",
            pilihan_stress="Kadang-kadang stres",
        ),
    },
    {
        "nama": "TEST 3 - Semua faktor risiko tinggi harus Tinggi",
        "expected_label": "Tinggi",
        "params": dict(
            usia=25,
            berat_kg=100,
            tinggi_cm=165,
            hba1c=7.0,
            fasting_blood_sugar=140,
            riwayat_diabetes_keluarga="Ada pada ayah/ibu/saudara kandung",
            physical_activity=0,
            dietary_habits=0,
            smoking=1,
            alcohol=1,
            pilihan_sleep="Kurang dari 5 jam",
            pilihan_stress="Sangat sering stres",
        ),
    },
    {
        "nama": "TEST 4 - Backward compatibility parameter lama tetap berjalan",
        "expected_min_label": "Sedang",
        "params": dict(
            usia=23,
            berat_kg=78,
            tinggi_cm=170,
            hba1c=6.0,
            fasting_blood_sugar=110,
            pilihan_genetic="Ada (ayah/ibu/saudara kandung)",
            family_history=1,
            physical_activity=0,
            dietary_habits=1,
            smoking=0,
            alcohol=0,
            pilihan_sleep="7-8 jam (ideal)",
            pilihan_stress="Kadang-kadang stres",
        ),
    },
]

rank = {
    "Tidak Berisiko": 0,
    "Sedang": 1,
    "Tinggi": 2,
}

semua_hasil = []

for tc in test_cases:
    print(f"\n{'=' * 60}")
    print(f"  {tc['nama']}")
    print(f"{'=' * 60}")

    hasil = prediksi_diabetes(**tc["params"])

    if "expected_score" in tc:
        assert hasil["risk_score"] == tc["expected_score"], (
            f"Skor salah. Diharapkan {tc['expected_score']}, "
            f"didapat {hasil['risk_score']}"
        )

    if "expected_label" in tc:
        assert hasil["scoring_label"] == tc["expected_label"], (
            f"Label salah. Diharapkan {tc['expected_label']}, "
            f"didapat {hasil['scoring_label']}"
        )

    if "expected_min_label" in tc:
        assert rank[hasil["scoring_label"]] >= rank[tc["expected_min_label"]], (
            f"Label terlalu rendah. Minimal {tc['expected_min_label']}, "
            f"didapat {hasil['scoring_label']}"
        )

    semua_hasil.append((tc["nama"], hasil))

print(f"\n{'=' * 60}")
print("  RINGKASAN SEMUA TEST")
print(f"{'=' * 60}")
print(
    f"  {'Kasus':<48} "
    f"{'Skor':<6} "
    f"{'Scoring':<18} "
    f"{'Model':<18} "
    f"{'Keyakinan'}"
)
print(
    f"  {'-' * 48} "
    f"{'-' * 6} "
    f"{'-' * 18} "
    f"{'-' * 18} "
    f"{'-' * 10}"
)

for nama, hasil in semua_hasil:
    label = nama.split(" - ", 1)[1] if " - " in nama else nama
    confidence = hasil["model_confidence_pct"]
    confidence_text = f"{confidence:.2f}%" if confidence is not None else "-"

    print(
        f"  {label:<48} "
        f"{hasil['risk_score']:<6} "
        f"{hasil['scoring_label']:<18} "
        f"{hasil['model_risk_label']:<18} "
        f"{confidence_text}"
    )

print(f"{'=' * 60}")


  TEST 1 - Contoh skor 7 harus Tidak Berisiko
          HASIL PREDIKSI RISIKO DIABETES
  Usia                       : 23 tahun
  Berat Badan                : 60 kg
  Tinggi Badan               : 165 cm
  BMI                        : 22.04 (Normal)
  HbA1c                      : 5.7%
  Gula Darah Puasa           : 100 mg/dL
  Riwayat Diabetes Keluarga  : Tidak ada keluarga diabetes

  Skor Risiko                : 7
  Kategori Scoring           : Tidak Berisiko
  Kategori Prediksi Model    : Tidak Berisiko
  Keyakinan Model            : 93.62%
  Catatan                    : Keyakinan model bukan probabilitas medis.

  Probabilitas per kelas:
    Tidak Berisiko    : 93.62%  ????????????????????????????
    Sedang            : 6.36%  ?
    Tinggi            : 0.02%  

          REKOMENDASI PERSONAL
  1. 🩸 HbA1c kamu di rentang prediabetes (5.7–6.4%). Kurangi konsumsi gula dan karbohidrat sederhana, pantau secara berkala.
  2. 🍬 Gula darah puasa kamu 100–125 mg/dL (prediabetes). Kurangi ka